In [0]:
df = spark.read.csv(
    "/Workspace/Users/vanshkrjain@gmail.com/data/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [0]:
from pyspark.sql.functions import col

df.filter(col('Sub-Category') == 'Appliances')\
    .select('Product ID', 'Profit')\
        .show()

+---------------+---------+
|     Product ID|   Profit|
+---------------+---------+
|OFF-AP-10002892|    34.47|
|OFF-AP-10002311| -123.858|
|OFF-AP-10001492|  15.6884|
|OFF-AP-10002118|  56.2032|
|OFF-AP-10000358|  22.5852|
|OFF-AP-10001058| 218.2518|
|OFF-AP-10000326|   17.766|
|OFF-AP-10002518| -453.849|
|OFF-AP-10000891|  -131.12|
|OFF-AP-10002684|  -243.16|
|OFF-AP-10003622|   0.2925|
|OFF-AP-10003217|-178.9668|
|OFF-AP-10002945| 496.0725|
|OFF-AP-10002203|   -4.466|
|OFF-AP-10002684|   24.316|
|OFF-AP-10001124| 168.4384|
|OFF-AP-10000696| -20.3322|
|OFF-AP-10004249|   9.6957|
|OFF-AP-10002472|   36.372|
|OFF-AP-10002578|   5.8704|
+---------------+---------+
only showing top 20 rows


In [0]:
df_revised = (df.withColumnRenamed("Postal Code","Zip Code")
.withColumn('Profit', col('Profit').cast('double'))
)

In [0]:
df_revised.printSchema()
df_revised.show()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zip Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+----

In [0]:
df_orders = df.filter(
    (col('Ship Mode') == 'Second Class') &
    (col('Profit') > 1000)
).show()

+------+--------------+----------+----------+------------+-----------+-------------+-----------+-------------+---------------+--------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|   Ship Mode|Customer ID|Customer Name|    Segment|      Country|           City|   State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+------------+-----------+-------------+-----------+-------------+---------------+--------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|  5563|CA-2017-133263|2017-03-31|2017-04-02|Second Class|   JE-15610|      Jim Epp|  Corporate|United States|        Atlanta| Georgia|      30318|  South|TEC-CO-10001449|     Technology|     Copiers|Hewlett Packard L...| 2

In [0]:
df_updated = df.withColumn(
    'Final Profit',
    col('Profit') * 1.18
)

In [0]:
df_updated.write.format('parquet')\
    .mode('overwrite')\
    .save('/Volumes/workspace/default/data_files/superstore_updated')

In [0]:
spark.read.parquet('/Volumes/workspace/default/data_files/superstore_updated')\
    .filter(col("Customer ID").isNotNull())\
    .write.mode('overwrite')\
    .option('header', True)\
    .csv('/Volumes/workspace/default/data_files/superstore_updated_csv')

In [0]:
df.filter(
    (col('Region') == 'North') |
    (col('Ship Mode') == 'First Class')
).show()

+------+--------------+----------+----------+-----------+-----------+------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|  Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+-----------+-----------+------------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|    36|CA-2016-117590|2016-12-08|2016-12-10|First Class|   GH-14485|         Gene Hale|  Corporate|United States|   Richardson|     Texas|      75080|Central|TEC-PH-10004977|     Technology|      Phones|         G